# 🧩 Mini-Lab: Structured Logging

**Module: Trust but Verify: Validation, Config, Errors, Logs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why structured (JSON) logs are superior to plain-text `print()` statements for debugging and monitoring
2. **Emit** structured log records with a run ID, timestamps, and per-record fields using Python's built-in `logging` module
3. **Redact** sensitive data (API keys and user text) in log output to maintain secrets hygiene

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Structured Logging | Logs as records with a run id and consistent per-record fields, so you can trace behavior and search reliably, while redacting sensitive keys and user text by default. |
| Secrets Hygiene | Secrets hygiene is the habit of keeping passwords, API keys, and tokens out of code and logs, sharing them safely, and rotating them regularly so one mistake doesn’t become a breach. |


## Why Structured Logging?

Plain-text logs like `print("Processing item 42...")` are easy to write but painful to work with at scale:

| Problem with `print()` | Structured logging solves it |
|------------------------|-----------------------------|
| No consistent format — every line is different | Every record is a JSON object with the same keys |
| Can't filter or search by field | Filter by `run_id`, `level`, `item_id`, etc. |
| Timestamps missing or inconsistent | Auto-generated ISO timestamps on every record |
| Secrets accidentally printed | A redaction filter strips them automatically |
| No correlation across a pipeline run | A `run_id` ties all records from one execution together |

## Step 1 — Set Up a JSON Log Formatter

Python's built-in `logging` module supports custom formatters. We'll create one that outputs each log record as a single JSON line — easy for humans to scan and trivial for tools to parse.

In [1]:
import json
import logging
import uuid
from datetime import datetime, timezone


class JSONFormatter(logging.Formatter):
    """Format every log record as a single JSON line."""

    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "message": record.getMessage(),
        }
        # Merge any extra fields passed via `extra={}`
        for key in ("run_id", "item_id", "model", "tokens", "latency_ms"):
            if hasattr(record, key):
                log_entry[key] = getattr(record, key)
        return json.dumps(log_entry)

## Step 2 — Create a Logger with the JSON Formatter

We wire the formatter into a standard Python logger. The `run_id` is generated once per execution and attached to every record so you can correlate all logs from a single pipeline run.

In [2]:
def get_logger(name: str = "app") -> logging.Logger:
    """Create (or retrieve) a logger that outputs JSON lines."""
    logger = logging.getLogger(name)

    # Avoid adding duplicate handlers on re-runs in a notebook
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(JSONFormatter())
        logger.addHandler(handler)
        logger.setLevel(logging.DEBUG)

    return logger


# Generate a run_id for this execution
run_id = uuid.uuid4().hex[:8]
logger = get_logger()

logger.info("Pipeline started", extra={"run_id": run_id})

{"timestamp": "2026-09-17T01:18:59.610863+00:00", "level": "INFO", "message": "Pipeline started", "run_id": "69f3ed31"}


Each log line is a valid JSON object with a timestamp, level, message, and the `run_id` that ties it to this run.

## Step 3 — Log with Per-Record Fields

Structured logging shines when each record carries **context fields** — the item being processed, the model used, token counts, latency. These fields make filtering and debugging easy.

In [3]:
import time

# Simulate processing a few items
items = [
    {"id": "doc-001", "text": "Summarize the quarterly earnings report."},
    {"id": "doc-002", "text": "Translate this paragraph to French."},
    {"id": "doc-003", "text": "Extract the key dates from the contract."},
]

for item in items:
    start = time.monotonic()
    # Simulate a model call
    time.sleep(0.05)
    latency = (time.monotonic() - start) * 1000

    logger.info(
        "Item processed",
        extra={
            "run_id": run_id,
            "item_id": item["id"],
            "model": "gpt-4o-mini",
            "tokens": 124,
            "latency_ms": round(latency, 1),
        },
    )

{"timestamp": "2026-09-17T01:18:59.683983+00:00", "level": "INFO", "message": "Item processed", "run_id": "69f3ed31", "item_id": "doc-001", "model": "gpt-4o-mini", "tokens": 124, "latency_ms": 47.0}


{"timestamp": "2026-09-17T01:18:59.736816+00:00", "level": "INFO", "message": "Item processed", "run_id": "69f3ed31", "item_id": "doc-002", "model": "gpt-4o-mini", "tokens": 124, "latency_ms": 63.0}


{"timestamp": "2026-09-17T01:18:59.788672+00:00", "level": "INFO", "message": "Item processed", "run_id": "69f3ed31", "item_id": "doc-003", "model": "gpt-4o-mini", "tokens": 124, "latency_ms": 47.0}


Every line has the same shape. You could pipe these into any log aggregator (Datadog, CloudWatch, ELK) and immediately filter by `run_id`, `item_id`, or `level`.

## Step 4 — Redact Secrets and User Text

Recall from `mini-settings-from-env`: secrets must never appear in logs. But it's easy to accidentally log a full API key or a user's private input. A **redaction filter** inspects every log record and masks sensitive values *before* they're written.

We'll redact two things:
1. **API keys** — any string matching common key prefixes (e.g., `sk-...`)
2. **User text** — truncated to a safe preview so raw user data doesn't fill log storage

In [4]:
import re


class RedactingFilter(logging.Filter):
    """Redact API keys and truncate user text in log records."""

    # Matches common API key patterns
    KEY_PATTERN = re.compile(r"(sk-|key-|token-)[A-Za-z0-9_-]{4,}")
    MAX_USER_TEXT = 30  # characters to keep before truncating

    def _redact_keys(self, text: str) -> str:
        """Replace API keys with a redacted placeholder."""
        return self.KEY_PATTERN.sub(
            lambda m: m.group()[:6] + "...REDACTED", text
        )

    def _truncate_user_text(self, text: str) -> str:
        """Shorten user text so raw inputs don't flood logs."""
        if len(text) > self.MAX_USER_TEXT:
            return text[: self.MAX_USER_TEXT] + "…[truncated]"
        return text

    def filter(self, record: logging.LogRecord) -> bool:
        # Redact the main message
        record.msg = self._redact_keys(str(record.msg))

        # Redact/truncate known extra fields
        if hasattr(record, "api_key"):
            record.api_key = self._redact_keys(str(record.api_key))
        if hasattr(record, "user_text"):
            record.user_text = self._truncate_user_text(str(record.user_text))

        return True  # always keep the record


# Attach the filter to our logger
logger.addFilter(RedactingFilter())
print("✅ RedactingFilter attached")

✅ RedactingFilter attached


## Step 5 — See Redaction in Action

Let's log some records that contain sensitive data and verify the filter cleans them up.

In [5]:
# --- API key redaction ---
logger.info(
    "Connecting with key sk-abc123realkey789xyz",
    extra={"run_id": run_id},
)

# --- User text truncation ---
long_user_input = (
    "Please summarize the following confidential document about our "
    "Q4 merger plans including all financial projections and personnel changes."
)

# We add user_text to the JSONFormatter's recognized fields for this demo
class JSONFormatterV2(JSONFormatter):
    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "message": record.getMessage(),
        }
        for key in ("run_id", "item_id", "model", "tokens",
                    "latency_ms", "api_key", "user_text"):
            if hasattr(record, key):
                log_entry[key] = getattr(record, key)
        return json.dumps(log_entry)

# Update the handler's formatter
logger.handlers[0].setFormatter(JSONFormatterV2())

logger.info(
    "Processing user request",
    extra={
        "run_id": run_id,
        "item_id": "req-042",
        "user_text": long_user_input,
    },
)

{"timestamp": "2026-09-17T01:18:59.830351+00:00", "level": "INFO", "message": "Connecting with key sk-abc...REDACTED", "run_id": "69f3ed31"}


{"timestamp": "2026-09-17T01:18:59.832351+00:00", "level": "INFO", "message": "Processing user request", "run_id": "69f3ed31", "item_id": "req-042", "user_text": "Please summarize the following\u2026[truncated]"}


Notice:
- The API key `sk-abc123realkey789xyz` became `sk-abc...REDACTED`
- The long user text was truncated to 30 characters + `…[truncated]`

The raw secrets and user content never reach the log output.

## Step 6 — Side-by-Side: `print()` vs. Structured Logging

In [6]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

md("""
| | ❌ `print()` | ✅ Structured logging |
|---|---|---|
| **Format** | `"Processing doc-001... done in 52ms"` | `{"item_id": "doc-001", "latency_ms": 52, ...}` |
| **Filtering** | `grep` and hope | Query by any field |
| **Correlation** | None — which run produced this line? | `run_id` on every record |
| **Secrets** | Accidentally printed if you're not careful | Redaction filter strips them automatically |
| **User text** | Full raw input in your logs | Truncated to a safe preview |
| **Timestamps** | Often missing | Auto-generated UTC ISO format |
""")


| | ❌ `print()` | ✅ Structured logging |
|---|---|---|
| **Format** | `"Processing doc-001... done in 52ms"` | `{"item_id": "doc-001", "latency_ms": 52, ...}` |
| **Filtering** | `grep` and hope | Query by any field |
| **Correlation** | None — which run produced this line? | `run_id` on every record |
| **Secrets** | Accidentally printed if you're not careful | Redaction filter strips them automatically |
| **User text** | Full raw input in your logs | Truncated to a safe preview |
| **Timestamps** | Often missing | Auto-generated UTC ISO format |


## 🎯 Summary

### Key Takeaways

1. **Structured logging** — emit log records as JSON objects with consistent fields (`timestamp`, `level`, `run_id`, `item_id`) so logs are machine-parseable and filterable, not free-form text
2. **Run ID correlation** — generate a unique `run_id` per execution and attach it to every log record so you can trace all events from a single pipeline run
3. **Per-record context** — pass contextual fields (model, tokens, latency) via `extra={}` to make each log line self-describing without string interpolation
4. **Redaction filter** — attach a `logging.Filter` that automatically masks API keys and truncates user text before records are written, so secrets never leak into log output